# Unidad 4 — Análisis de Agrupamientos (Clustering)

> **Inteligencia Computacional · USACH · Prof. Max Chacón**  
> Notebook de ejercicios: jerárquico, k-means, DBSCAN y métricas de calidad.

**Temas cubiertos:**
1. Clustering jerárquico con dendrograma (single/complete/average/Ward)
2. k-means con inicialización k-means++ y método del codo
3. DBSCAN: detección de formas arbitrarias y ruido
4. Índices de calidad: Silhouette, Davies-Bouldin, Calinski-Harabasz
5. Tabla comparativa de métodos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.datasets import load_breast_cancer, make_moons, make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score,
    calinski_harabasz_score, adjusted_rand_score
)
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110
rng = np.random.default_rng(42)

## 1. Clustering jerárquico — ejemplo manual (Chacón, Cap. IV)

5 puntos en $\mathbb{R}^2$: A(1,1), B(1,2), C(2,1), D(5,4), E(5,5).

In [ ]:
puntos = np.array([[1,1],[1,2],[2,1],[5,4],[5,5]], dtype=float)
nombres = list('ABCDE')

# Matriz de distancias
D_mat = squareform(pdist(puntos, metric='euclidean'))
df_dist = pd.DataFrame(np.round(D_mat, 2), index=nombres, columns=nombres)
print('Matriz de distancias euclidiana:')
print(df_dist.to_string())

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
linkages = ['single', 'complete', 'average', 'ward']

for ax, method in zip(axes, linkages):
    Z = linkage(puntos, method=method)
    dendrogram(Z, labels=nombres, ax=ax, color_threshold=0)
    ax.set_title(f'Linkage: {method}')
    ax.set_ylabel('Distancia')

plt.suptitle('Dendrogramas con distintos criterios de enlace', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

## 2. k-means — método del codo

In [ ]:
# Dataset con 3 clusters gaussianos
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=0)

wcss = []
K_range = range(1, 10)
for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X_blobs)
    wcss.append(km.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(K_range, wcss, 'o-', lw=2, color='steelblue')
axes[0].axvline(3, color='tomato', ls='--', label='k=3 (codo)')
axes[0].set_xlabel('k'); axes[0].set_ylabel('WCSS (Inercia)')
axes[0].set_title('Método del Codo'); axes[0].legend()

km_final = KMeans(n_clusters=3, init='k-means++', n_init=10, random_state=42)
labels_km = km_final.fit_predict(X_blobs)
centroids = km_final.cluster_centers_

scatter = axes[1].scatter(X_blobs[:,0], X_blobs[:,1], c=labels_km, cmap='Set2', s=30, alpha=0.8)
axes[1].scatter(centroids[:,0], centroids[:,1], c='black', marker='X', s=200, label='Centroides', zorder=5)
axes[1].set_title('k-means (k=3) — resultado final'); axes[1].legend()

plt.tight_layout(); plt.show()

print(f'ARI vs verdad: {adjusted_rand_score(y_blobs, labels_km):.3f}')

## 3. DBSCAN — clusters de forma arbitraria

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# k-means (falla en lunas)
labels_km2 = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(X_moons)
axes[0].scatter(X_moons[:,0], X_moons[:,1], c=labels_km2, cmap='Set1', s=20)
axes[0].set_title('k-means (falla en formas no convexas)')

# DBSCAN
db = DBSCAN(eps=0.2, min_samples=5)
labels_db = db.fit_predict(X_moons)
n_noise = (labels_db == -1).sum()
axes[1].scatter(X_moons[:,0], X_moons[:,1], c=labels_db, cmap='Set1', s=20)
axes[1].set_title(f'DBSCAN (eps=0.2, min_pts=5) — ruido: {n_noise}')

# Verdad
axes[2].scatter(X_moons[:,0], X_moons[:,1], c=y_moons, cmap='Set1', s=20)
axes[2].set_title('Verdad')

plt.tight_layout(); plt.show()
print(f'DBSCAN ARI: {adjusted_rand_score(y_moons, labels_db):.3f}')

## 4. Métricas de calidad — Wisconsin Breast Cancer

In [ ]:
data = load_breast_cancer()
X_bc = StandardScaler().fit_transform(data.data)
y_bc = data.target

# Reducir a 2D con PCA para visualización
X_2d = PCA(n_components=2).fit_transform(X_bc)

resultados = []
for method_name, model in [
    ('k-means k=2', KMeans(n_clusters=2, n_init=10, random_state=42)),
    ('Ward k=2',    AgglomerativeClustering(n_clusters=2, linkage='ward')),
    ('Complete k=2',AgglomerativeClustering(n_clusters=2, linkage='complete')),
]:
    lbl = model.fit_predict(X_bc)
    resultados.append({
        'Método': method_name,
        'Silhouette': silhouette_score(X_bc, lbl),
        'Davies-Bouldin': davies_bouldin_score(X_bc, lbl),
        'Calinski-Harabasz': calinski_harabasz_score(X_bc, lbl),
        'ARI (vs clase real)': adjusted_rand_score(y_bc, lbl),
    })

df_res = pd.DataFrame(resultados).set_index('Método')
print('Comparación de métodos (Wisconsin Breast Cancer):')
print(df_res.round(4).to_string())

In [ ]:
# Silhouette por punto para k-means
from sklearn.metrics import silhouette_samples

km2 = KMeans(n_clusters=2, n_init=10, random_state=42)
labels_bc = km2.fit_predict(X_bc)
sil_vals = silhouette_samples(X_bc, labels_bc)

fig, ax = plt.subplots(figsize=(8, 4))
y_lower = 10
for i in range(2):
    vals = np.sort(sil_vals[labels_bc == i])
    y_upper = y_lower + len(vals)
    color = plt.cm.Set2(i / 2)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, vals, facecolor=color, alpha=0.8)
    ax.text(-0.05, y_lower + 0.5 * len(vals), f'Cluster {i}')
    y_lower = y_upper + 10
ax.axvline(sil_vals.mean(), color='red', ls='--', label=f'Media={sil_vals.mean():.3f}')
ax.set_xlabel('Coeficiente Silhouette')
ax.set_title('Silhouette por punto — k-means k=2 (Wisconsin BC)')
ax.legend()
plt.tight_layout(); plt.show()

## 5. Selección de k con múltiples métricas

In [ ]:
metrics_k = {'k': [], 'Silhouette': [], 'Davies-Bouldin': [], 'Calinski-Harabasz': []}

for k in range(2, 9):
    lbl_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_bc)
    metrics_k['k'].append(k)
    metrics_k['Silhouette'].append(silhouette_score(X_bc, lbl_k))
    metrics_k['Davies-Bouldin'].append(davies_bouldin_score(X_bc, lbl_k))
    metrics_k['Calinski-Harabasz'].append(calinski_harabasz_score(X_bc, lbl_k))

df_m = pd.DataFrame(metrics_k).set_index('k')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col, better in zip(axes, df_m.columns, ['↑ mayor', '↓ menor', '↑ mayor']):
    ax.plot(df_m.index, df_m[col], 'o-', lw=2)
    ax.set_xlabel('k'); ax.set_ylabel(col)
    ax.set_title(f'{col}  ({better})')
plt.suptitle('Métricas de calidad vs k — k-means (Wisconsin BC)', y=1.02)
plt.tight_layout(); plt.show()

## Resumen

| Método | Fortaleza | Limitación |
|---|---|---|
| k-means | Rápido, escalable | Clusters esféricos, sensible a init |
| Ward | Clusters compactos, jerárquico | $O(n^2)$, outliers afectan |
| DBSCAN | Formas arbitrarias, detecta outliers | Sensible a $\varepsilon$ / MinPts |

| Métrica | Mejor valor | Interpreta |
|---|---|---|
| Silhouette | $\uparrow$ (max 1) | Cohesión + separación |
| Davies-Bouldin | $\downarrow$ (min 0) | Ratio dispersión intra / inter |
| Calinski-Harabasz | $\uparrow$ | BSS / WSS |

> **Ejercicio propuesto**: sobre el dataset del L3, comparar k-means (k=2,3,4) y Ward con las 3 métricas. Reportar en una tabla y justificar la elección de k.